## Dataset Process

In [ ]:
from datasets import load_from_disk, Dataset, DatasetDict
import os
from nmr_rise.utils.data import dataset_aug_molref, conformation_construction
import json
import copy
import ast

In [ ]:
from datasets import load_dataset
import os
import pandas as pd

In [ ]:
file_name_list = os.listdir('../../raw_data/USPTO-NMR-790K/multi_spectra_data/')
os.makedirs('../../raw_data/USPTO-NMR-790K/nmr_spectra_data/', exist_ok=True)
for file_name in file_name_list:
    file_path = os.path.join('../../raw_data/USPTO-NMR-790K/multi_spectra_data/', file_name)
    df = pd.read_parquet(file_path, columns=['smiles', 'molecular_formula', 'h_nmr_peaks', 'c_nmr_peaks'])
    # only save columns 'smiles', 'h_nmr_peaks', 'c_nmr_peaks', 'molecular_formula'
    df.to_parquet(os.path.join('../../raw_data/USPTO-NMR-790K/nmr_spectra_data/', file_name), index=False)


In [ ]:
import pandas as pd

df = pd.read_parquet('../../raw_data/USPTO-NMR-790K/nmr_peak_data/aligned_chunk_0.parquet')

In [ ]:
import pandas as pd
import os
from tqdm import tqdm
file_names = os.listdir('../../raw_data/USPTO-NMR-790K/nmr_peak_data/')
df = pd.DataFrame(columns=['smiles', 'file_name', 'index'])
data_list = []
for file in tqdm(file_names):
    file_path = os.path.join('../../raw_data/USPTO-NMR-790K/nmr_peak_data/', file)
    df_temp = pd.read_parquet(file_path)
    for index, row in df_temp.iterrows():
        data_list.append({'smiles': row['smiles'], 'file_name': file, 'index': index})
df = pd.DataFrame(data_list)
df.to_csv(os.path.join('../../raw_data/USPTO-NMR-790K/smiles_index.csv'), index=False)

## NMR2Mol Dataset Generation

In [ ]:
from datasets import load_dataset
from nmr_rise.utils.data import filter_invalid_smiles
from datasets import DatasetDict
dataset = load_dataset('parquet', data_files='../../raw_data/USPTO-NMR-790K/nmr_spectra_data/*.parquet')['train']
print("Dataset size before filtering:", len(dataset))
dataset = dataset.filter(filter_invalid_smiles, num_proc=16)
print("Dataset size after filtering:", len(dataset))
# split dataset into train, val, test with 80% train, 10% val, 10% test
def cal_max_shift(entry):
    cnmr_values = [peak['delta (ppm)'] for peak in entry['c_nmr_peaks']]
    hnmr_values = [peak['delta'] for peak in entry['h_nmr_peaks']]
    max_c_shift = max(cnmr_values) if cnmr_values else 0
    max_h_shift = max(hnmr_values) if hnmr_values else 0
    min_c_shift = min(cnmr_values) if cnmr_values else 0
    min_h_shift = min(hnmr_values) if hnmr_values else 0
    entry['max_c_shift'] = max_c_shift
    entry['max_h_shift'] = max_h_shift
    entry['min_c_shift'] = min_c_shift
    entry['min_h_shift'] = min_h_shift
    return entry
dataset = dataset.map(cal_max_shift, num_proc=16)
# print("Dataset size before filtering:", len(dataset['train']))
dataset = dataset.filter(lambda x: x['max_c_shift'] <= 250 and x['max_h_shift'] <= 15, num_proc=16)
dataset = dataset.filter(lambda x: x['min_c_shift'] >= -20 and x['min_h_shift'] >= -2, num_proc=16)
print("Dataset size after filtering:", len(dataset))
train_test_split = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = train_test_split['train']
test_valid_dataset = train_test_split['test']
val_test_split = test_valid_dataset.train_test_split(test_size=0.5, seed=42)
val_dataset = val_test_split['train']
test_dataset = val_test_split['test']
dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})
dataset.save_to_disk('../../data/USPTO-NMR-790K/full')

## Mol2NMR Dataset Generation

In [ ]:
from datasets import load_from_disk
from nmr_rise.utils.data import conformation_construction, filter_invalid_entry
dataset = load_from_disk('../../data/USPTO-NMR-790K/full')
dataset.cleanup_cache_files()
for split in ['train', 'validation', 'test']:
    dataset[split] = dataset[split].map(conformation_construction, num_proc=16)
    dataset[split] = dataset[split].filter(filter_invalid_entry, num_proc=16)
dataset.save_to_disk('../../data/USPTO-NMR-790K/full_cc')

## NMR2Mol Dataset Generation

In [ ]:
import os
from datasets import load_from_disk
import json
import ast
import copy
def load_nmr2mol_pred_results(dir_path, filename):
    with open(os.path.join(dir_path, filename), 'r') as f:
        return {item['true']: str(item['pred']) for item in json.load(f)}

def add_candidates(entry, lookup, validity_check=True):
    candidates = ast.literal_eval(lookup.get(entry['smiles'], ""))

    if validity_check:
        valid_candidates = []
        for cand in candidates:
            confs = conformation_construction({'smiles': copy.deepcopy(cand)})
            if confs['is_converted']:
                valid_candidates.append(cand)
        entry['candidates'] = valid_candidates
    else:
        entry['candidates'] = candidates
    return entry

def molref_data_gen(data_path, dataset_name, nmr2mol_pred_dir):
    dataset = load_from_disk(os.path.join(data_path, dataset_name))
    nmr2mol_pred_path = os.path.join(data_path, nmr2mol_pred_dir)
    lookups = {
        'train': load_nmr2mol_pred_results(nmr2mol_pred_path, 'train_split_result.json'),
        'validation': load_nmr2mol_pred_results(nmr2mol_pred_path, 'valid_split_result.json'),
        'test': load_nmr2mol_pred_results(nmr2mol_pred_path, 'test_split_result.json')
    }
    for split in ['train', 'validation', 'test']:
        dataset[split] = dataset[split].map(lambda ex: add_candidates(ex, lookups[split], validity_check=False), num_proc=16)
    return dataset

In [ ]:
dataset = molref_data_gen('../../data/USPTO-NMR-790K', 'full', 'nmr2mol_pred')

In [ ]:
import random
from ast import literal_eval
def candidate_aug(batch, aug_num=5):
    """
    Batch version of candidate augmentation function.
    For each example in the batch, samples aug_num candidates from the 'candidates' list,
    then expands each example into multiple rows - one per sampled candidate.
    
    Args:
        batch (dict): A batch of examples as lists for each column.
        aug_num (int): Number of candidates to sample and expand per example.
    
    Returns:
        dict: Augmented batch with each example expanded into multiple rows.
    """
    aug_rows = {
        k: [] for k in batch if k != 'candidates'
    }
    aug_rows['candidate'] = []
    aug_rows['aug_id'] = []
    
    for i, candidates in enumerate(batch['candidates']):
        # Sample candidates with or without replacement
        sampled_candidates = random.sample(candidates, aug_num)
        
        for j, candidate in enumerate(sampled_candidates):
            # Copy all columns except 'candidates'
            for key in batch:
                if key != 'candidates':
                    aug_rows[key].append(batch[key][i])
            aug_rows['candidate'].append(candidate)
            aug_rows['aug_id'].append(f"aug_{batch.get('idx', [''])[i]}_{j}")

    return aug_rows

def dataset_aug_molref(dataset, aug_num=5, aug_max=False):
    """
    Augments the dataset for training the MolRef model by expanding each example into multiple rows based on candidates.
    
    Args:
        dataset (datasets.Dataset): HuggingFace dataset to augment.
        aug_num (int): Number of augmented samples per example.
    
    Returns:
        datasets.Dataset: Augmented dataset with expanded rows and extra columns.
    """
    return dataset.map(
        lambda batch: candidate_aug(batch, aug_num=len(batch['candidates'][0]) if aug_max else min(aug_num, len(batch['candidates'][0]))),
        batched=True,
        batch_size=1,  # Process one example at a time for expansion
        remove_columns=['candidates'],
        num_proc=os.cpu_count()
    )


In [ ]:
from datasets import DatasetDict
for aug_num in [1, 3, 5, 10]:
    shuffle_seed = 42
    train_aug = dataset_aug_molref(dataset['train'], aug_num=aug_num, aug_max=False)
    val_aug = dataset_aug_molref(dataset['validation'], aug_num=aug_num, aug_max=False)
    test_aug = dataset_aug_molref(dataset['test'], aug_num=aug_num, aug_max=False)
    train_aug = train_aug.shuffle(seed=shuffle_seed)
    val_aug = val_aug.shuffle(seed=shuffle_seed)
    test_aug = test_aug.shuffle(seed=shuffle_seed)
    aug_dataset = DatasetDict({
        'train': train_aug,
        'validation': val_aug,
        'test': test_aug
    })
    aug_dataset.save_to_disk(os.path.join('../../data/USPTO-NMR-790K', f'revision_{aug_num}'))

# Evaluation Dataset Generation

In [ ]:
from datasets import load_from_disk
import random

test_dataset = load_from_disk('../../data/USPTO-NMR/full')['test']

# randomly select 1000 samples from test_dataset
random.seed(42)
selected_indices = random.sample(range(len(test_dataset)), 10000)
selected_samples = test_dataset.select(selected_indices)
selected_samples.save_to_disk('../../data/USPTO-NMR/USPTO-NMR-10000')

selected_indices = random.sample(range(len(selected_samples)), 1000)
selected_samples = selected_samples.select(selected_indices)
selected_samples.save_to_disk('../../data/USPTO-NMR/USPTO-NMR-1000')